# Download Dataset

In [275]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# For time series
from typing import List
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import userdata

In [276]:
def download_kaggle(kaggle_command="kaggle competitions download -c liver-fibrosis-severity-prediction"):

  # Get Kaggle Key
  kaggle_username = userdata.get("KAGGLE_USER")
  kaggle_key = userdata.get("KAGGLE_KEY")
  if not kaggle_username or not kaggle_key:
      print("Error: Kaggle_USERNAME or Kaggle_KEY not found in Colab Secrets.")
      return

  # Write the credentials to ~/.kaggle/kaggle.json
  kaggle_dir = os.path.expanduser("~/.kaggle")
  os.makedirs(kaggle_dir, exist_ok=True)

  # Create JSON
  kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
  with open(kaggle_json_path, "w") as f:
      f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')
  os.chmod(kaggle_json_path, 0o600)

  try:
      os.system(kaggle_command)
      print("\n--- Download complete! ---")
      os.system("ls -la")
      os.system("unzip -o '*.zip' && rm -f *.zip")
      os.system("ls -la")

  except Exception as e:
      print(f"An error occurred during download: {e}")

In [277]:
download_kaggle("kaggle competitions download -c individual-test-spai-sorting-hat")


--- Download complete! ---


# Explore Dataset

## Import Data

In [278]:
df_attendance = pd.read_csv("/content/attendance.csv")
df_attendance

,user_id,datetime
0,5931fa6a-af6b-43b8-babb-28349854d406,2024-04-19 17:30:08.827555
1,5f1a2ecf-943d-446b-b94d-d7d1ad64213d,2024-04-19 17:30:11.405020
2,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,2024-04-19 17:30:14.433098
3,cdd098ee-4295-4839-b1a3-4d44840feacd,2024-04-19 17:30:17.214453
4,f922a5d4-3651-46f7-aa98-307c045dd4b5,2024-04-19 17:30:21.556805
...,...,...
10798,ef836255-9ff1-48bd-9ee1-b3e205f96367,2024-05-17 13:57:11.458189
10799,84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f,2024-05-17 13:57:15.457449
10800,1e103917-1d29-424d-b5e1-924e0cdcd6dc,2024-05-17 13:57:19.924906
10801,9207b26d-79dc-41fc-b62c-67a6821132b9,2024-05-17 14:01:03.323543


In [279]:
df_house = pd.read_csv("/content/train.csv")
df_house

,user_id,house
0,4b2569be-cd32-40aa-9cc4-9e13e55903de,house1
1,cdd098ee-4295-4839-b1a3-4d44840feacd,house3
2,226c662f-b1b3-40bc-9bfc-4f49a80b9041,house6
3,7889a31f-7351-4b4d-b872-e04e5e35b9dd,house2
4,0492584f-5dda-4ad6-ab2f-79c8d43a8753,house1
...,...,...
97,7b3f7524-f201-44df-a929-5b92eac9457b,house5
98,6f16696b-22a0-4f3c-b2ad-f6b9e0256775,house3
99,272f295e-c673-4c17-8172-b9342b9d1192,house6
100,1f7d7fe6-a09f-45c9-87f1-5bec07a066fa,house2


In [280]:
df_submission = pd.read_csv("/content/submission.csv")
df_submission

,user_id,house
0,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3
1,81c9f456-b5a2-4e28-ad69-eb7d89374d37,house1
2,dda733a2-2add-44a5-b888-81943923c2c6,house4
3,12584309-4699-485d-9f5b-a57ebfc4283a,NaN
4,c968fef9-5411-49d5-b6e1-d9a52add4ab8,NaN
...,...,...
63,fed3dced-4009-461e-9411-a440c779e32c,NaN
64,60fea3c0-9066-49ea-beb2-b055ab2b4ff9,NaN
65,1bedb648-5a50-4680-a09b-27240759f9a0,NaN
66,ef836255-9ff1-48bd-9ee1-b3e205f96367,NaN


## Explore Dataset

In [281]:
def extract_time_features(df: pd.DataFrame, time_columns: list = None) -> pd.DataFrame:
    """
    Extracts time-based features from specified columns in a DataFrame.
    If time_columns is not provided, it attempts to infer time columns
    by checking common time-related column names.

    Args:
        df (pd.DataFrame): The input DataFrame.
        time_columns (list, optional): A list of column names containing time data.
                                        If None, common time-related column names will be checked.

    Returns:
        pd.DataFrame: The DataFrame with new time feature columns.
    """

    df_copy = df.copy()

    # Common time-related column name patterns for inference
    if time_columns is None:
        possible_time_col_patterns = [
            'time', 'date', 'timestamp', 'datetime', 'created_at',
            'updated_at', 'ts', 'event_time', 'start_time', 'end_time'
        ]

        inferred_time_columns = [
            col for col in df_copy.columns
            if any(pattern in col.lower() for pattern in possible_time_col_patterns)
        ]

        if not inferred_time_columns:
            print("Warning: No time columns specified and no common time-related columns found. "
                  "Returning original DataFrame.")
            return df_copy
        time_columns = inferred_time_columns
        print(f"Inferred time columns: {time_columns}")

    print(f"{time_columns} is valid")


    # Possible time formats to try for robust parsing
    possible_time_formats = [
        "%Y-%m-%d %H:%M:%S.%f",  # 2025-07-05 13:42:29.123456
        "%Y-%m-%d %H:%M:%S",    # 2025-07-05 13:42:29
        "%Y-%m-%dT%H:%M:%S.%fZ",# ISO 8601 with Z (UTC)
        "%Y-%m-%dT%H:%M:%S",    # ISO 8601 without Z
        "%Y/%m/%d %H:%M:%S",    # 2025/07/05 13:42:29
        "%d-%m-%Y %H:%M:%S",    # 05-07-2025 13:42:29
        "%m/%d/%Y %I:%M:%S %p", # 07/05/2025 01:42:29 PM
        "%d/%m/%Y %H:%M",       # 05/07/2025 13:42
        "%Y-%m-%d",             # 2025-07-05
        "%d-%m-%Y",             # 05-07-2025
        "%m/%d/%Y",             # 07/05/2025
        "%H:%M:%S",             # 13:42:29
        "%H:%M",                # 13:42
        "%I:%M %p"              # 01:42 PM
    ]

    for col in time_columns:
        if col not in df_copy.columns:
            print(f"Warning: Column '{col}' not found in DataFrame. Skipping.")
            continue

        print("Column is okay")
        print(col)

        # Convert to datetime, coercing errors to NaT (Not a Time)
        # We try multiple formats with errors='coerce' to handle mixed formats
        df_copy[f'{col}_dt'] = pd.to_datetime(df_copy[col], errors='coerce', infer_datetime_format=True)
        print(df_copy)

        # Drop rows where conversion failed for this column, or handle as NaT
        # For simplicity in feature extraction, we'll proceed with NaT values
        # If a column is entirely NaT after conversion, it's probably not a time column
        if df_copy[f'{col}_dt'].isnull().all() and not df_copy[col].isnull().all():
            print(f"Warning: Column '{col}' could not be converted to datetime. Skipping feature extraction for this column.")
            df_copy = df_copy.drop(columns=[f'{col}_dt'])
            continue

        # Extract features
        dt_col = df_copy[f'{col}_dt'].dt
        df_copy[f'{col}_year'] = dt_col.year
        df_copy[f'{col}_month'] = dt_col.month
        df_copy[f'{col}_day'] = dt_col.day
        df_copy[f'{col}_hour'] = dt_col.hour
        df_copy[f'{col}_minute'] = dt_col.minute
        df_copy[f'{col}_second'] = dt_col.second
        df_copy[f'{col}_dayofweek'] = dt_col.dayofweek # Monday=0, Sunday=6
        df_copy[f'{col}_dayofyear'] = dt_col.dayofyear
        df_copy[f'{col}_quarter'] = dt_col.quarter
        df_copy[f'{col}_is_weekend'] = dt_col.dayofweek.isin([5, 6]).astype(int)
        df_copy[f'{col}_season'] = (dt_col.month % 12 + 3) // 3 # Simple season mapping
        df_copy[f'{col}_is_month_start'] = dt_col.is_month_start.astype(int)
        df_copy[f'{col}_is_month_end'] = dt_col.is_month_end.astype(int)
        df_copy[f'{col}_is_quarter_start'] = dt_col.is_quarter_start.astype(int)
        df_copy[f'{col}_is_quarter_end'] = dt_col.is_quarter_end.astype(int)

        # Optional: Add cyclical features (sin/cos transformations for hour, dayofweek, etc.)
        df_copy[f'{col}_hour_sin'] = np.sin(2 * np.pi * dt_col.hour / 24)
        df_copy[f'{col}_hour_cos'] = np.cos(2 * np.pi * dt_col.hour / 24)

    return df_copy

In [282]:
df_attendance = extract_time_features(df=df_attendance, time_columns=["datetime"])
df_attendance

['datetime'] is valid
Column is okay
datetime
                                    user_id                    datetime  \
0      5931fa6a-af6b-43b8-babb-28349854d406  2024-04-19 17:30:08.827555   
1      5f1a2ecf-943d-446b-b94d-d7d1ad64213d  2024-04-19 17:30:11.405020   
2      6e009a96-2b8b-4048-8b0c-20a7924a4b4f  2024-04-19 17:30:14.433098   
3      cdd098ee-4295-4839-b1a3-4d44840feacd  2024-04-19 17:30:17.214453   
4      f922a5d4-3651-46f7-aa98-307c045dd4b5  2024-04-19 17:30:21.556805   
...                                     ...                         ...   
10798  ef836255-9ff1-48bd-9ee1-b3e205f96367  2024-05-17 13:57:11.458189   
10799  84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f  2024-05-17 13:57:15.457449   
10800  1e103917-1d29-424d-b5e1-924e0cdcd6dc  2024-05-17 13:57:19.924906   
10801  9207b26d-79dc-41fc-b62c-67a6821132b9  2024-05-17 14:01:03.323543   
10802  25e69851-885b-4cd1-8c32-e29e604714ea  2024-05-17 14:01:05.950781   

                     datetime_dt  
0     2024-04-19 1

/tmp/ipython-input-281-1860511102.py:68: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_copy[f'{col}_dt'] = pd.to_datetime(df_copy[col], errors='coerce', infer_datetime_format=True)


,user_id,datetime,datetime_dt,datetime_year,datetime_month,datetime_day,datetime_hour,datetime_minute,datetime_second,datetime_dayofweek,datetime_dayofyear,datetime_quarter,datetime_is_weekend,datetime_season,datetime_is_month_start,datetime_is_month_end,datetime_is_quarter_start,datetime_is_quarter_end,datetime_hour_sin,datetime_hour_cos
0,5931fa6a-af6b-43b8-babb-28349854d406,2024-04-19 17:30:08.827555,2024-04-19 17:30:08.827555,2024.0,4.0,19.0,17.0,30.0,8.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
1,5f1a2ecf-943d-446b-b94d-d7d1ad64213d,2024-04-19 17:30:11.405020,2024-04-19 17:30:11.405020,2024.0,4.0,19.0,17.0,30.0,11.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
2,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,2024-04-19 17:30:14.433098,2024-04-19 17:30:14.433098,2024.0,4.0,19.0,17.0,30.0,14.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
3,cdd098ee-4295-4839-b1a3-4d44840feacd,2024-04-19 17:30:17.214453,2024-04-19 17:30:17.214453,2024.0,4.0,19.0,17.0,30.0,17.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
4,f922a5d4-3651-46f7-aa98-307c045dd4b5,2024-04-19 17:30:21.556805,2024-04-19 17:30:21.556805,2024.0,4.0,19.0,17.0,30.0,21.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10798,ef836255-9ff1-48bd-9ee1-b3e205f96367,2024-05-17 13:57:11.458189,2024-05-17 13:57:11.458189,2024.0,5.0,17.0,13.0,57.0,11.0,4.0,138.0,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926
10799,84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f,2024-05-17 13:57:15.457449,2024-05-17 13:57:15.457449,2024.0,5.0,17.0,13.0,57.0,15.0,4.0,138.0,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926
10800,1e103917-1d29-424d-b5e1-924e0cdcd6dc,2024-05-17 13:57:19.924906,2024-05-17 13:57:19.924906,2024.0,5.0,17.0,13.0,57.0,19.0,4.0,138.0,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926
10801,9207b26d-79dc-41fc-b62c-67a6821132b9,2024-05-17 14:01:03.323543,2024-05-17 14:01:03.323543,2024.0,5.0,17.0,14.0,1.0,3.0,4.0,138.0,2.0,0,2.0,0,0,0,0,-0.500000,-0.866025


In [283]:
df_attendance_house = pd.merge(df_attendance, df_house, how="inner", on=["user_id"])
df_attendance_house = df_attendance_house.dropna()
df_attendance_house

,user_id,datetime,datetime_dt,datetime_year,datetime_month,datetime_day,datetime_hour,datetime_minute,datetime_second,datetime_dayofweek,...,datetime_quarter,datetime_is_weekend,datetime_season,datetime_is_month_start,datetime_is_month_end,datetime_is_quarter_start,datetime_is_quarter_end,datetime_hour_sin,datetime_hour_cos,house
0,cdd098ee-4295-4839-b1a3-4d44840feacd,2024-04-19 17:30:17.214453,2024-04-19 17:30:17.214453,2024.0,4.0,19.0,17.0,30.0,17.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819,house3
1,2ad18754-2522-4fa9-af12-bccdf42d3905,2024-04-19 17:30:24.008379,2024-04-19 17:30:24.008379,2024.0,4.0,19.0,17.0,30.0,24.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819,house5
2,3bd6dcac-904c-4614-be50-9c3949cb3405,2024-04-19 17:30:26.610108,2024-04-19 17:30:26.610108,2024.0,4.0,19.0,17.0,30.0,26.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819,house5
3,1d9dd772-17c9-4c8c-a3dd-1dbfe41c5ba9,2024-04-19 17:30:29.393161,2024-04-19 17:30:29.393161,2024.0,4.0,19.0,17.0,30.0,29.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819,house5
4,7eecd605-6ca1-4343-bf40-56ab7ea20dc9,2024-04-19 17:30:43.212998,2024-04-19 17:30:43.212998,2024.0,4.0,19.0,17.0,30.0,43.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819,house3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6479,7a92cb13-de00-4cf9-8f2e-fe1de7629692,2024-05-17 13:57:03.521711,2024-05-17 13:57:03.521711,2024.0,5.0,17.0,13.0,57.0,3.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926,house4
6480,3179b05a-487a-4208-9d7a-115c9532149b,2024-05-17 13:57:06.084968,2024-05-17 13:57:06.084968,2024.0,5.0,17.0,13.0,57.0,6.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926,house4
6481,84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f,2024-05-17 13:57:15.457449,2024-05-17 13:57:15.457449,2024.0,5.0,17.0,13.0,57.0,15.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926,house3
6482,9207b26d-79dc-41fc-b62c-67a6821132b9,2024-05-17 14:01:03.323543,2024-05-17 14:01:03.323543,2024.0,5.0,17.0,14.0,1.0,3.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.500000,-0.866025,house3


In [284]:
df_attendance_house_submission = df_submission.merge(df_attendance, how='inner', on='user_id')
df_attendance_house_submission

,user_id,house,datetime,datetime_dt,datetime_year,datetime_month,datetime_day,datetime_hour,datetime_minute,datetime_second,...,datetime_dayofyear,datetime_quarter,datetime_is_weekend,datetime_season,datetime_is_month_start,datetime_is_month_end,datetime_is_quarter_start,datetime_is_quarter_end,datetime_hour_sin,datetime_hour_cos
0,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3,2024-04-19 17:30:14.433098,2024-04-19 17:30:14.433098,2024.0,4.0,19.0,17.0,30.0,14.0,...,110.0,2.0,0,2.0,0,0,0,0,-9.659258e-01,-0.258819
1,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3,2024-04-20 08:39:08.013977,2024-04-20 08:39:08.013977,2024.0,4.0,20.0,8.0,39.0,8.0,...,111.0,2.0,1,2.0,0,0,0,0,8.660254e-01,-0.500000
2,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3,2024-04-20 12:48:46.776276,2024-04-20 12:48:46.776276,2024.0,4.0,20.0,12.0,48.0,46.0,...,111.0,2.0,1,2.0,0,0,0,0,1.224647e-16,-1.000000
3,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3,2024-04-20 17:38:35.927190,2024-04-20 17:38:35.927190,2024.0,4.0,20.0,17.0,38.0,35.0,...,111.0,2.0,1,2.0,0,0,0,0,-9.659258e-01,-0.258819
4,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3,2024-04-21 08:36:59.624088,2024-04-21 08:36:59.624088,2024.0,4.0,21.0,8.0,36.0,59.0,...,112.0,2.0,1,2.0,0,0,0,0,8.660254e-01,-0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4314,7474c4b5-d72d-4213-ad48-c8376a66cf35,NaN,2024-05-15 17:43:44.548400,2024-05-15 17:43:44.548400,2024.0,5.0,15.0,17.0,43.0,44.0,...,136.0,2.0,0,2.0,0,0,0,0,-9.659258e-01,-0.258819
4315,7474c4b5-d72d-4213-ad48-c8376a66cf35,NaN,2024-05-16 08:43:56.181531,2024-05-16 08:43:56.181531,2024.0,5.0,16.0,8.0,43.0,56.0,...,137.0,2.0,0,2.0,0,0,0,0,8.660254e-01,-0.500000
4316,7474c4b5-d72d-4213-ad48-c8376a66cf35,NaN,2024-05-16 12:51:12.472693,2024-05-16 12:51:12.472693,2024.0,5.0,16.0,12.0,51.0,12.0,...,137.0,2.0,0,2.0,0,0,0,0,1.224647e-16,-1.000000
4317,7474c4b5-d72d-4213-ad48-c8376a66cf35,NaN,2024-05-16 17:30:30.331964,2024-05-16 17:30:30.331964,2024.0,5.0,16.0,17.0,30.0,30.0,...,137.0,2.0,0,2.0,0,0,0,0,-9.659258e-01,-0.258819


In [285]:
df_attendance_house_submission.columns

Index(['user_id', 'house', 'datetime', 'datetime_dt', 'datetime_year',
       'datetime_month', 'datetime_day', 'datetime_hour', 'datetime_minute',
       'datetime_second', 'datetime_dayofweek', 'datetime_dayofyear',
       'datetime_quarter', 'datetime_is_weekend', 'datetime_season',
       'datetime_is_month_start', 'datetime_is_month_end',
       'datetime_is_quarter_start', 'datetime_is_quarter_end',
       'datetime_hour_sin', 'datetime_hour_cos'],
      dtype='object')

In [286]:
df_attendance_house = df_attendance_house.drop(
    columns=[
        "datetime", "datetime_month",
        "datetime_year", "datetime_quarter", "datetime_season",
        "datetime_is_month_start", "datetime_is_month_end", "datetime_is_quarter_start",
        "datetime_is_quarter_end", 'datetime_dayofweek', 'datetime_dayofyear',
       'datetime_is_weekend', 'datetime_hour_sin', 'datetime_hour_cos'
    ]
)


df_attendance_house_submission = df_attendance_house_submission.drop(
    columns=[
        "datetime", "datetime_month",
        "datetime_year", "datetime_quarter", "datetime_season",
        "datetime_is_month_start", "datetime_is_month_end", "datetime_is_quarter_start",
        "datetime_is_quarter_end", 'datetime_dayofweek', 'datetime_dayofyear',
       'datetime_is_weekend', 'datetime_hour_sin', 'datetime_hour_cos'
    ]
)

In [287]:
df_attendance_house

,user_id,datetime_dt,datetime_day,datetime_hour,datetime_minute,datetime_second,house
0,cdd098ee-4295-4839-b1a3-4d44840feacd,2024-04-19 17:30:17.214453,19.0,17.0,30.0,17.0,house3
1,2ad18754-2522-4fa9-af12-bccdf42d3905,2024-04-19 17:30:24.008379,19.0,17.0,30.0,24.0,house5
2,3bd6dcac-904c-4614-be50-9c3949cb3405,2024-04-19 17:30:26.610108,19.0,17.0,30.0,26.0,house5
3,1d9dd772-17c9-4c8c-a3dd-1dbfe41c5ba9,2024-04-19 17:30:29.393161,19.0,17.0,30.0,29.0,house5
4,7eecd605-6ca1-4343-bf40-56ab7ea20dc9,2024-04-19 17:30:43.212998,19.0,17.0,30.0,43.0,house3
...,...,...,...,...,...,...,...
6479,7a92cb13-de00-4cf9-8f2e-fe1de7629692,2024-05-17 13:57:03.521711,17.0,13.0,57.0,3.0,house4
6480,3179b05a-487a-4208-9d7a-115c9532149b,2024-05-17 13:57:06.084968,17.0,13.0,57.0,6.0,house4
6481,84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f,2024-05-17 13:57:15.457449,17.0,13.0,57.0,15.0,house3
6482,9207b26d-79dc-41fc-b62c-67a6821132b9,2024-05-17 14:01:03.323543,17.0,14.0,1.0,3.0,house3


In [288]:
df_attendance_house_submission

,user_id,house,datetime_dt,datetime_day,datetime_hour,datetime_minute,datetime_second
0,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3,2024-04-19 17:30:14.433098,19.0,17.0,30.0,14.0
1,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3,2024-04-20 08:39:08.013977,20.0,8.0,39.0,8.0
2,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3,2024-04-20 12:48:46.776276,20.0,12.0,48.0,46.0
3,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3,2024-04-20 17:38:35.927190,20.0,17.0,38.0,35.0
4,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3,2024-04-21 08:36:59.624088,21.0,8.0,36.0,59.0
...,...,...,...,...,...,...,...
4314,7474c4b5-d72d-4213-ad48-c8376a66cf35,NaN,2024-05-15 17:43:44.548400,15.0,17.0,43.0,44.0
4315,7474c4b5-d72d-4213-ad48-c8376a66cf35,NaN,2024-05-16 08:43:56.181531,16.0,8.0,43.0,56.0
4316,7474c4b5-d72d-4213-ad48-c8376a66cf35,NaN,2024-05-16 12:51:12.472693,16.0,12.0,51.0,12.0
4317,7474c4b5-d72d-4213-ad48-c8376a66cf35,NaN,2024-05-16 17:30:30.331964,16.0,17.0,30.0,30.0


In [289]:
df_attendance_house_submission["house"].value_counts()

,count
house,
house1,68
house3,66
house4,49


In [290]:
df_attendance_house_submission = df_attendance_house_submission.sort_values(by="datetime_dt")
df_attendance_house = df_attendance_house.sort_values(by="datetime_dt")

In [293]:
house_submission_list = []

for index_submit, row_submit in df_attendance_house_submission.iterrows():

  # # If not nan then skip
  # value = float('nan')
  # if not pd.isna(row_submit["house"]):
  #   print("House not empty")
  #   continue

  # Get values
  second = row_submit["datetime_minute"]
  minute = row_submit["datetime_second"]
  hour = row_submit["datetime_hour"]
  day = row_submit["datetime_day"]

  # Set RANGE for second
  SECOND_RANGE = 10

  # If nan, then we will loop check to the all available attendance
  df_range_same = df_attendance_house[
        (df_attendance_house["datetime_hour"] == hour) &
        (df_attendance_house["datetime_minute"] == minute) &
        (df_attendance_house["datetime_day"] == day) &
        (df_attendance_house["datetime_second"] <= (second + SECOND_RANGE)) &
        (df_attendance_house["datetime_second"] >= (second - SECOND_RANGE))
  ]


  # Find the most frequency in the range
  most_frequency_house = df_range_same["house"].mode()


  # Check if it empty or not
  if not most_frequency_house.empty:

    most_frequency_house = most_frequency_house.iloc[0]
    print(f"Most Frequency in 10 second is {most_frequency_house}")
    print(type(most_frequency_house))

    # row_submit["house"] = [most_frequency_house]
    # df_attendance_house_submission.loc[index_submit, ['house']] = [row_submit['house'], most_frequency_house]
    house_submission_list.append(most_frequency_house)

    print(row_submit["house"])

    # df.loc[index, ['B', 'C']] = [row['B'] * 10, 'new_value']


  else :

    # Set RANGE for second
    MINUTE_RANGE = 30

    # If nan, then we will loop check to the all available attendance
    df_range_same = df_attendance_house[
          (df_attendance_house["datetime_hour"] == hour) &
          (df_attendance_house["datetime_minute"] == (minute + MINUTE_RANGE)) &
          (df_attendance_house["datetime_minute"] == (minute - MINUTE_RANGE)) &
          (df_attendance_house["datetime_day"] == day)
    ]

    # Find the most frequency in the range
    most_frequency_house = df_range_same["house"].mode()

    # Check if it empty or not
    if not most_frequency_house.empty:
      most_frequency_house = most_frequency_house.iloc[0]
      print(f"Most Frequency in 30 minute is {most_frequency_house}")
      #row_submit["house"] = [most_frequency_house]
      house_submission_list.append(most_frequency_house)
      print(row_submit["house"])

    else :
      # Set RANGE for second
      MINUTE_RANGE = 60

      # If nan, then we will loop check to the all available attendance
      df_range_same = df_attendance_house[
            (df_attendance_house["datetime_hour"] == hour) &
            (df_attendance_house["datetime_minute"] == (minute + MINUTE_RANGE)) &
            (df_attendance_house["datetime_minute"] == (minute - MINUTE_RANGE)) &
            (df_attendance_house["datetime_day"] == day)
      ]

      # Find the most frequency in the range
      most_frequency_house = df_range_same["house"].mode()

      # Check if it empty or not
      if not most_frequency_house.empty:

        most_frequency_house = most_frequency_house.iloc[0]
        print(f"Most Frequency in 30 minute is {most_frequency_house}")

        #row_submit["house"] = [most_frequency_house]
        house_submission_list.append(most_frequency_house)
        print(row_submit["house"])

      else:
        print("Still not in range")
        #row_submit["house"] = "house1"
        house_submission_list.append("house1")

Streaming output truncated to the last 5000 lines.
Most Frequency in 10 second is house1
<class 'str'>
nan
Still not in range
Still not in range
Still not in range
Most Frequency in 10 second is house2
<class 'str'>
nan
Most Frequency in 10 second is house1
<class 'str'>
nan
Still not in range
Most Frequency in 10 second is house1
<class 'str'>
nan
Still not in range
Still not in range
Most Frequency in 10 second is house2
<class 'str'>
nan
Most Frequency in 10 second is house1
<class 'str'>
nan
Still not in range
Still not in range
Still not in range
Still not in range
Most Frequency in 10 second is house3
<class 'str'>
nan
Still not in range
Still not in range
Still not in range
Still not in range
Still not in range
Still not in range
Still not in range
Still not in range
Most Frequency in 10 second is house2
<class 'str'>
nan
Most Frequency in 10 second is house1
<class 'str'>
nan
Most Frequency in 10 second is house1
<class 'str'>
nan
Most Frequency in 10 second is house2
<class 's

In [294]:
house_submission_list

['house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house6',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house4',
 'house1',
 'house1',
 'house1',
 'house5',
 'house6',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house4',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house4',
 'house4',
 'house1',
 'house1',
 'house1',
 'house1',
 'house3',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',
 'house1',